# Exploratory Data Analysis: Steve Biko Hospital Emergency Department Arrivals

## Master's Thesis: Forecasting Patient Daily Arrivals for Hospital Resource Planning

---

**Author:** [Your Name]  
**Date:** March 2026  
**Institution:** [Your University]

---

## Abstract

This notebook presents a comprehensive exploratory data analysis (EDA) of Emergency Department (ED) arrival patterns at Steve Biko Academic Hospital, Pretoria, South Africa. The analysis spans four interconnected datasets covering patient arrivals, weather conditions, calendar features, and clinical presentations from 2019-2026. The primary objective is to develop insights that inform the construction of machine learning models for forecasting daily patient arrivals, ultimately supporting optimal staff scheduling and inventory management.

**Key Research Questions:**
1. What temporal patterns characterize ED arrivals at Steve Biko Hospital?
2. How do external factors (weather, holidays, weekends) influence patient volume?
3. What is the distribution of clinical presentations, and how does it vary temporally?
4. Which features are most predictive for forecasting future arrivals?

---

## 1. Introduction and Research Context

### 1.1 Background

Emergency Department overcrowding is a global healthcare challenge with significant implications for patient outcomes, staff wellbeing, and hospital operational efficiency (Morley et al., 2018). Accurate forecasting of patient arrivals enables proactive resource allocation, reducing wait times and improving care quality.

### 1.2 South African Healthcare Context

Steve Biko Academic Hospital is a tertiary teaching hospital in Pretoria, serving a diverse population. Key contextual factors include:

- **High trauma burden**: South Africa has elevated rates of interpersonal violence and road traffic accidents
- **Seasonal disease patterns**: Respiratory infections peak in winter (June-August)
- **Public health system strain**: Resource constraints necessitate efficient planning
- **Socioeconomic factors**: Income disparities affect healthcare-seeking behavior

### 1.3 Objectives of This EDA

This analysis aims to:
1. **Characterize data quality** - Identify missing values, outliers, and data integrity issues
2. **Understand distributions** - Examine univariate and multivariate patterns
3. **Identify temporal dynamics** - Discover seasonality, trends, and autocorrelation
4. **Quantify relationships** - Assess correlations between features and target variables
5. **Inform feature engineering** - Guide creation of predictive features for ML models
6. **Support model selection** - Determine appropriate forecasting approaches

---

## 2. Environment Setup and Data Loading

In [ ]:
# =============================================================================
# 2.1 Import Libraries
# =============================================================================

# Core data manipulation
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistical analysis
from scipy import stats
from scipy.stats import pearsonr, spearmanr, shapiro, normaltest
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.3f}'.format)

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Libraries loaded successfully")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
# =============================================================================
# 2.2 Define Data Paths
# =============================================================================

DATA_PATHS = {
    "patient": r"C:\Users\BIBINBUSINESS\OneDrive\Desktop\data transformation pipline\healthforecast_pipeline\healthforecast_pipeline\intermediate\raw_extracted\all_daily_extracted.csv",
    "weather": r"C:\Users\BIBINBUSINESS\OneDrive\Desktop\data steve biko\external factor data\pretoria_weather_2019_2026.csv",
    "calendar": r"C:\Users\BIBINBUSINESS\OneDrive\Desktop\data steve biko\external factor data\calendar_features_2019_2026.csv",
    "clinical": r"C:\Users\BIBINBUSINESS\OneDrive\Desktop\data transformation pipline\healthforecast_pipeline\healthforecast_pipeline\intermediate\clinical\clinical_reasons_daily.csv",
}

# Hospital metadata
HOSPITAL_NAME = "Steve Biko Academic Hospital"
LOCATION = "Pretoria, Gauteng, South Africa"
COORDINATES = (-25.7461, 28.1881)  # Latitude, Longitude

In [ ]:
# =============================================================================
# 2.3 Load Datasets
# =============================================================================

def load_dataset(path: str, name: str) -> pd.DataFrame:
    """Load a dataset with error handling and basic info display."""
    try:
        df = pd.read_csv(path)
        print(f"\n{'='*60}")
        print(f"Dataset: {name.upper()}")
        print(f"{'='*60}")
        print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
        print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        return df
    except FileNotFoundError:
        print(f"ERROR: File not found - {path}")
        return pd.DataFrame()

# Load all datasets
df_patient = load_dataset(DATA_PATHS["patient"], "Patient Arrivals")
df_weather = load_dataset(DATA_PATHS["weather"], "Weather")
df_calendar = load_dataset(DATA_PATHS["calendar"], "Calendar")
df_clinical = load_dataset(DATA_PATHS["clinical"], "Clinical Reasons")

---

## 3. Data Quality Assessment

### Justification for ML Modeling

Data quality directly impacts model performance. Missing values, outliers, and inconsistencies can introduce bias, reduce predictive accuracy, and lead to unreliable forecasts. This section systematically evaluates data integrity across all four datasets.

**Key Metrics:**
- Completeness (missing values)
- Consistency (data type validation)
- Accuracy (outlier detection)
- Temporal coverage (date range verification)

In [ ]:
# =============================================================================
# 3.1 Missing Values Analysis
# =============================================================================

def analyze_missing_values(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Comprehensive missing value analysis."""
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    
    analysis = pd.DataFrame({
        'Column': missing.index,
        'Missing': missing.values,
        'Percentage': missing_pct.values,
        'Non-Missing': len(df) - missing.values,
        'Dtype': df.dtypes.values
    })
    
    analysis = analysis.sort_values('Percentage', ascending=False)
    
    print(f"\n{'='*60}")
    print(f"Missing Values Analysis: {name}")
    print(f"{'='*60}")
    print(f"Total rows: {len(df):,}")
    print(f"Columns with missing values: {(missing > 0).sum()}")
    print(f"Total missing cells: {missing.sum():,} ({100*missing.sum()/(len(df)*len(df.columns)):.2f}%)")
    
    if (missing > 0).any():
        print("\nColumns with missing values:")
        print(analysis[analysis['Missing'] > 0].to_string(index=False))
    else:
        print("\n✓ No missing values detected")
    
    return analysis

# Analyze each dataset
missing_patient = analyze_missing_values(df_patient, "Patient Arrivals")
missing_weather = analyze_missing_values(df_weather, "Weather")
missing_calendar = analyze_missing_values(df_calendar, "Calendar")
missing_clinical = analyze_missing_values(df_clinical, "Clinical Reasons")

In [ ]:
# =============================================================================
# 3.2 Missing Values Visualization
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

datasets = [
    (df_patient, "Patient Arrivals"),
    (df_weather, "Weather"),
    (df_calendar, "Calendar"),
    (df_clinical, "Clinical Reasons")
]

for ax, (df, name) in zip(axes.flat, datasets):
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=True)
    
    if len(missing_pct) > 0:
        # Show top 15 columns with most missing
        if len(missing_pct) > 15:
            missing_pct = missing_pct.tail(15)
        missing_pct.plot(kind='barh', ax=ax, color='coral')
        ax.set_xlabel('Missing Percentage (%)')
        ax.set_title(f'{name}: Missing Values')
        ax.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='5% threshold')
    else:
        ax.text(0.5, 0.5, 'No Missing Values', ha='center', va='center', fontsize=14)
        ax.set_title(f'{name}: Missing Values')
        ax.axis('off')

plt.tight_layout()
plt.suptitle('Missing Values Analysis Across Datasets', fontsize=16, y=1.02)
plt.savefig('figures/missing_values_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("THESIS INSIGHT: Missing Values")
print("="*60)
print("""
For time-series forecasting, missing values require careful handling:
- Forward-fill: Appropriate for slowly-changing variables (e.g., weather)
- Interpolation: Suitable for continuous variables with clear trends
- Imputation: Model-based approaches for complex patterns
- Exclusion: For rows with critical missing target variables

Columns with >5% missing data require documented handling strategies.
""")

In [ ]:
# =============================================================================
# 3.3 Temporal Coverage Analysis
# =============================================================================

def analyze_temporal_coverage(df: pd.DataFrame, date_col: str, name: str):
    """Analyze temporal coverage and identify gaps."""
    if date_col not in df.columns:
        print(f"Date column '{date_col}' not found in {name}")
        return
    
    df_temp = df.copy()
    df_temp[date_col] = pd.to_datetime(df_temp[date_col], errors='coerce')
    df_temp = df_temp.dropna(subset=[date_col]).sort_values(date_col)
    
    date_range = df_temp[date_col]
    
    print(f"\n{'='*60}")
    print(f"Temporal Coverage: {name}")
    print(f"{'='*60}")
    print(f"Start Date: {date_range.min().strftime('%Y-%m-%d')}")
    print(f"End Date: {date_range.max().strftime('%Y-%m-%d')}")
    print(f"Total Days: {(date_range.max() - date_range.min()).days:,}")
    print(f"Unique Dates: {date_range.nunique():,}")
    print(f"Records: {len(df_temp):,}")
    
    # Check for gaps
    expected_dates = pd.date_range(start=date_range.min(), end=date_range.max(), freq='D')
    actual_dates = set(date_range.dt.date)
    missing_dates = set(expected_dates.date) - actual_dates
    
    if missing_dates:
        print(f"\n⚠ Missing Dates: {len(missing_dates)}")
        if len(missing_dates) <= 10:
            for d in sorted(missing_dates):
                print(f"  - {d}")
        else:
            print(f"  (First 5: {sorted(list(missing_dates))[:5]})")
    else:
        print("\n✓ Complete daily coverage (no gaps)")
    
    # Duplicates check
    duplicates = df_temp[date_col].duplicated().sum()
    if duplicates > 0:
        print(f"\n⚠ Duplicate dates: {duplicates}")
    else:
        print("✓ No duplicate dates")

# Analyze temporal coverage for each dataset
analyze_temporal_coverage(df_patient, 'date', 'Patient Arrivals')
analyze_temporal_coverage(df_weather, 'date', 'Weather')
analyze_temporal_coverage(df_calendar, 'date', 'Calendar')
analyze_temporal_coverage(df_clinical, 'date', 'Clinical Reasons')

In [ ]:
# =============================================================================
# 3.4 Data Type Validation
# =============================================================================

def validate_data_types(df: pd.DataFrame, name: str):
    """Validate and summarize data types."""
    type_counts = df.dtypes.value_counts()
    
    print(f"\n{'='*60}")
    print(f"Data Types: {name}")
    print(f"{'='*60}")
    
    for dtype, count in type_counts.items():
        cols = df.select_dtypes(include=[dtype]).columns.tolist()
        print(f"\n{dtype}: {count} columns")
        if len(cols) <= 10:
            print(f"  Columns: {cols}")
        else:
            print(f"  Columns: {cols[:5]} ... (+{len(cols)-5} more)")

validate_data_types(df_patient, "Patient Arrivals")
validate_data_types(df_weather, "Weather")
validate_data_types(df_calendar, "Calendar")
validate_data_types(df_clinical, "Clinical Reasons")

---

## 4. Univariate Analysis

### Justification for ML Modeling

Understanding individual variable distributions is essential for:
1. **Feature scaling decisions** - Identifying skewed distributions requiring transformation
2. **Outlier detection** - Finding anomalous values that may affect model training
3. **Baseline understanding** - Establishing expected ranges for validation
4. **Model selection** - Distributions inform choice of loss functions and algorithms

In [ ]:
# =============================================================================
# 4.1 Patient Arrivals - Target Variable Analysis
# =============================================================================

# Identify the main target variable (patient count)
print("\n" + "="*60)
print("PATIENT ARRIVALS: Column Overview")
print("="*60)
print(df_patient.columns.tolist())

# Key patient count column
target_col = 'patient_count'
if target_col not in df_patient.columns:
    # Try alternative names
    alternatives = ['total_arrivals', 'arrivals', 'count', 'patients']
    for alt in alternatives:
        if alt in df_patient.columns:
            target_col = alt
            break
    else:
        # Use first numeric column
        numeric_cols = df_patient.select_dtypes(include=['number']).columns
        if len(numeric_cols) > 0:
            target_col = numeric_cols[0]

print(f"\nTarget Variable: {target_col}")

In [ ]:
# =============================================================================
# 4.2 Target Variable Distribution
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Get target data
if target_col in df_patient.columns:
    target_data = df_patient[target_col].dropna()
    
    # Histogram with KDE
    ax1 = axes[0, 0]
    sns.histplot(target_data, kde=True, ax=ax1, color='steelblue', edgecolor='white')
    ax1.axvline(target_data.mean(), color='red', linestyle='--', label=f'Mean: {target_data.mean():.1f}')
    ax1.axvline(target_data.median(), color='green', linestyle='--', label=f'Median: {target_data.median():.1f}')
    ax1.set_title(f'Distribution of Daily Patient Arrivals')
    ax1.set_xlabel('Patient Count')
    ax1.legend()
    
    # Box plot
    ax2 = axes[0, 1]
    bp = ax2.boxplot(target_data, vert=True, patch_artist=True)
    bp['boxes'][0].set_facecolor('steelblue')
    ax2.set_title('Box Plot: Patient Arrivals')
    ax2.set_ylabel('Patient Count')
    
    # Q-Q Plot for normality assessment
    ax3 = axes[1, 0]
    stats.probplot(target_data, dist="norm", plot=ax3)
    ax3.set_title('Q-Q Plot: Normality Assessment')
    
    # Time series plot
    ax4 = axes[1, 1]
    df_patient_temp = df_patient.copy()
    df_patient_temp['date'] = pd.to_datetime(df_patient_temp['date'], errors='coerce')
    df_patient_temp = df_patient_temp.sort_values('date')
    ax4.plot(df_patient_temp['date'], df_patient_temp[target_col], linewidth=0.5, alpha=0.7)
    ax4.set_title('Time Series: Daily Patient Arrivals')
    ax4.set_xlabel('Date')
    ax4.set_ylabel('Patient Count')
    ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('figures/target_variable_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistical summary
print("\n" + "="*60)
print("TARGET VARIABLE STATISTICS")
print("="*60)
print(f"Count: {len(target_data):,}")
print(f"Mean: {target_data.mean():.2f}")
print(f"Median: {target_data.median():.2f}")
print(f"Std Dev: {target_data.std():.2f}")
print(f"Min: {target_data.min():.0f}")
print(f"Max: {target_data.max():.0f}")
print(f"Skewness: {target_data.skew():.3f}")
print(f"Kurtosis: {target_data.kurtosis():.3f}")
print(f"\nPercentiles:")
print(f"  5th: {target_data.quantile(0.05):.0f}")
print(f"  25th: {target_data.quantile(0.25):.0f}")
print(f"  75th: {target_data.quantile(0.75):.0f}")
print(f"  95th: {target_data.quantile(0.95):.0f}")

# Normality tests
if len(target_data) < 5000:
    shapiro_stat, shapiro_p = shapiro(target_data)
    print(f"\nShapiro-Wilk Test: W={shapiro_stat:.4f}, p={shapiro_p:.4e}")
else:
    print("\n(Shapiro-Wilk skipped - sample size > 5000)")

dagostino_stat, dagostino_p = normaltest(target_data)
print(f"D'Agostino-Pearson Test: K²={dagostino_stat:.4f}, p={dagostino_p:.4e}")

print("\n" + "="*60)
print("THESIS INSIGHT: Target Variable")
print("="*60)
print("""
Key findings for model selection:
- If skewness > 1: Consider log transformation or Poisson-based models
- If heavy tails (high kurtosis): Robust loss functions may be needed
- Non-normal distribution: Tree-based models (XGBoost) may outperform linear models
- Variance-to-mean ratio informs choice between Poisson and Negative Binomial
""")

In [ ]:
# =============================================================================
# 4.3 Patient Arrivals - All Numeric Variables
# =============================================================================

# Descriptive statistics for all numeric columns
print("\n" + "="*60)
print("PATIENT ARRIVALS: Descriptive Statistics")
print("="*60)

numeric_patient = df_patient.select_dtypes(include=['number'])
print(numeric_patient.describe().T.round(2))

In [ ]:
# =============================================================================
# 4.4 Weather Variables Distribution
# =============================================================================

print("\n" + "="*60)
print("WEATHER DATA: Column Overview")
print("="*60)
print(df_weather.columns.tolist())

# Descriptive statistics
print("\n" + "="*60)
print("WEATHER DATA: Descriptive Statistics")
print("="*60)
numeric_weather = df_weather.select_dtypes(include=['number'])
print(numeric_weather.describe().T.round(2))

In [ ]:
# =============================================================================
# 4.5 Weather Variables Visualization
# =============================================================================

# Key weather columns to visualize
weather_cols = ['temp_mean_C', 'temp_max_C', 'temp_min_C', 'precipitation_mm', 
                'wind_max_kmh', 'humidity_pct']
weather_cols = [c for c in weather_cols if c in df_weather.columns]

if weather_cols:
    n_cols = min(len(weather_cols), 3)
    n_rows = (len(weather_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    
    for i, col in enumerate(weather_cols):
        data = df_weather[col].dropna()
        sns.histplot(data, kde=True, ax=axes[i], color='teal')
        axes[i].axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.1f}')
        axes[i].set_title(f'{col}')
        axes[i].legend()
    
    # Hide unused subplots
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.suptitle('Weather Variables Distribution (Pretoria)', fontsize=14, y=1.02)
    plt.savefig('figures/weather_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# =============================================================================
# 4.6 Clinical Reasons Distribution
# =============================================================================

print("\n" + "="*60)
print("CLINICAL REASONS: Column Overview")
print("="*60)
print(f"Total columns: {len(df_clinical.columns)}")
print(f"\nColumn names:")
for i, col in enumerate(df_clinical.columns):
    print(f"  {i+1}. {col}")

In [ ]:
# =============================================================================
# 4.7 Clinical Categories - Key Variables
# =============================================================================

# Identify key clinical columns
clinical_key_cols = [col for col in df_clinical.columns if any(x in col.lower() for x in 
    ['trauma', 'violence', 'spec_', 'total_', 'priority', 'triage'])]

print("\n" + "="*60)
print("KEY CLINICAL VARIABLES: Descriptive Statistics")
print("="*60)

if clinical_key_cols:
    key_clinical = df_clinical[clinical_key_cols].select_dtypes(include=['number'])
    print(key_clinical.describe().T.round(2))

In [ ]:
# =============================================================================
# 4.8 Specialty Distribution Visualization
# =============================================================================

# Find specialty columns
spec_cols = [col for col in df_clinical.columns if col.startswith('spec_')]

if spec_cols:
    spec_totals = df_clinical[spec_cols].sum().sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(spec_totals)))
    spec_totals.plot(kind='barh', ax=ax, color=colors)
    ax.set_xlabel('Total Patients (All Days)')
    ax.set_title('Patient Distribution by Medical Specialty')
    
    # Add value labels
    for i, v in enumerate(spec_totals.values):
        ax.text(v + 1000, i, f'{v:,.0f}', va='center')
    
    plt.tight_layout()
    plt.savefig('figures/specialty_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*60)
    print("SPECIALTY BREAKDOWN")
    print("="*60)
    total_all = spec_totals.sum()
    for spec, count in spec_totals.sort_values(ascending=False).items():
        pct = 100 * count / total_all
        print(f"{spec}: {count:,.0f} ({pct:.1f}%)")

In [ ]:
# =============================================================================
# 4.9 Trauma and Violence Analysis (South African Context)
# =============================================================================

print("\n" + "="*60)
print("TRAUMA AND VIOLENCE ANALYSIS")
print("="*60)
print("\nContext: South Africa has one of the world's highest rates of")
print("interpersonal violence, impacting ED case mix significantly.")

# Find trauma/violence columns
trauma_cols = [col for col in df_clinical.columns if any(x in col.lower() for x in 
    ['trauma', 'violence', 'assault', 'injury', 'stab', 'gunshot', 'accident'])]

if trauma_cols:
    print(f"\nTrauma-related columns found: {len(trauma_cols)}")
    
    # Summary statistics
    trauma_df = df_clinical[trauma_cols].select_dtypes(include=['number'])
    print("\nDescriptive Statistics:")
    print(trauma_df.describe().T.round(2))
    
    # Total trauma cases
    trauma_totals = trauma_df.sum().sort_values(ascending=False)
    print("\nTotal Cases (All Days):")
    for col, count in trauma_totals.items():
        print(f"  {col}: {count:,.0f}")

---

## 5. Temporal Pattern Analysis

### Justification for ML Modeling

Time-series forecasting requires understanding temporal dependencies:
1. **Trends** - Long-term directional movements inform baseline models
2. **Seasonality** - Periodic patterns guide feature engineering (day-of-week, month)
3. **Autocorrelation** - Lag dependencies determine optimal lookback windows
4. **Cyclic patterns** - Non-fixed cycles (e.g., pay days) require calendar features

In [ ]:
# =============================================================================
# 5.1 Prepare Time-Series Data
# =============================================================================

# Create unified time-indexed dataset
df_ts = df_patient.copy()
df_ts['date'] = pd.to_datetime(df_ts['date'], errors='coerce')
df_ts = df_ts.dropna(subset=['date']).sort_values('date').set_index('date')

print(f"Time series range: {df_ts.index.min()} to {df_ts.index.max()}")
print(f"Total days: {(df_ts.index.max() - df_ts.index.min()).days}")
print(f"Records: {len(df_ts)}")

In [ ]:
# =============================================================================
# 5.2 Day-of-Week Pattern
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Create day of week
df_ts['day_of_week'] = df_ts.index.dayofweek
df_ts['day_name'] = df_ts.index.day_name()

# Order days
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

if target_col in df_ts.columns:
    # Box plot by day of week
    dow_data = df_ts.groupby('day_name')[target_col].apply(list).reindex(day_order)
    
    bp = axes[0].boxplot([dow_data[d] for d in day_order], labels=day_order, patch_artist=True)
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, 7))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    axes[0].set_title('Patient Arrivals by Day of Week')
    axes[0].set_ylabel('Patient Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Mean with confidence interval
    dow_stats = df_ts.groupby('day_name')[target_col].agg(['mean', 'std', 'count']).reindex(day_order)
    dow_stats['se'] = dow_stats['std'] / np.sqrt(dow_stats['count'])
    dow_stats['ci95'] = 1.96 * dow_stats['se']
    
    axes[1].bar(range(7), dow_stats['mean'], yerr=dow_stats['ci95'], 
                color=colors, capsize=5, edgecolor='black')
    axes[1].set_xticks(range(7))
    axes[1].set_xticklabels(day_order, rotation=45)
    axes[1].set_title('Mean Arrivals by Day (95% CI)')
    axes[1].set_ylabel('Mean Patient Count')

plt.tight_layout()
plt.savefig('figures/day_of_week_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("DAY-OF-WEEK STATISTICS")
print("="*60)
print(dow_stats[['mean', 'std', 'count']].round(2))

# ANOVA test for day-of-week effect
from scipy.stats import f_oneway
groups = [df_ts[df_ts['day_name'] == d][target_col].values for d in day_order]
f_stat, p_value = f_oneway(*groups)
print(f"\nOne-way ANOVA: F={f_stat:.2f}, p={p_value:.4e}")
print(f"Significant day-of-week effect: {'Yes' if p_value < 0.05 else 'No'}")

In [ ]:
# =============================================================================
# 5.3 Monthly Seasonality
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_ts['month'] = df_ts.index.month
df_ts['month_name'] = df_ts.index.month_name()

month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

if target_col in df_ts.columns:
    # Box plot by month
    month_data = df_ts.groupby('month_name')[target_col].apply(list).reindex(month_order)
    
    bp = axes[0].boxplot([month_data[m] for m in month_order], labels=[m[:3] for m in month_order], 
                         patch_artist=True)
    colors = plt.cm.coolwarm(np.linspace(0, 1, 12))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    axes[0].set_title('Patient Arrivals by Month')
    axes[0].set_ylabel('Patient Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Mean with trend line
    month_stats = df_ts.groupby('month')[target_col].agg(['mean', 'std']).reset_index()
    axes[1].plot(month_stats['month'], month_stats['mean'], 'o-', linewidth=2, markersize=8)
    axes[1].fill_between(month_stats['month'], 
                         month_stats['mean'] - month_stats['std'],
                         month_stats['mean'] + month_stats['std'],
                         alpha=0.3)
    axes[1].set_xticks(range(1, 13))
    axes[1].set_xticklabels([m[:3] for m in month_order])
    axes[1].set_title('Monthly Mean Arrivals (±1 SD)')
    axes[1].set_xlabel('Month')
    axes[1].set_ylabel('Mean Patient Count')
    
    # Mark winter months (June-August in Southern Hemisphere)
    axes[1].axvspan(6, 8, alpha=0.2, color='blue', label='Winter (flu season)')
    axes[1].legend()

plt.tight_layout()
plt.savefig('figures/monthly_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("THESIS INSIGHT: Seasonal Patterns")
print("="*60)
print("""
South African Seasonal Context:
- Winter (June-August): Peak respiratory infections, flu season
- Summer (December-February): Holiday trauma, outdoor activities
- December: Holiday season with increased alcohol-related incidents
- Month-end: Pay day effects on healthcare-seeking behavior

Feature Engineering Implications:
- Include month as categorical or cyclical feature (sin/cos encoding)
- Create 'is_winter' binary flag for respiratory model
- Consider holiday season indicator for December
""")

In [ ]:
# =============================================================================
# 5.4 Year-over-Year Comparison
# =============================================================================

df_ts['year'] = df_ts.index.year

if target_col in df_ts.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Annual totals
    yearly_stats = df_ts.groupby('year')[target_col].agg(['sum', 'mean', 'std', 'count'])
    
    # Bar chart of annual totals
    axes[0].bar(yearly_stats.index, yearly_stats['sum'], color='steelblue', edgecolor='black')
    axes[0].set_title('Total Annual Patient Arrivals')
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Total Patients')
    
    for i, (year, val) in enumerate(zip(yearly_stats.index, yearly_stats['sum'])):
        axes[0].text(year, val + 500, f'{val:,.0f}', ha='center', va='bottom', fontsize=9)
    
    # Daily mean by year
    axes[1].bar(yearly_stats.index, yearly_stats['mean'], 
                yerr=yearly_stats['std'], capsize=5,
                color='teal', edgecolor='black')
    axes[1].set_title('Mean Daily Arrivals by Year (±1 SD)')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Mean Daily Patients')
    
    plt.tight_layout()
    plt.savefig('figures/yearly_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*60)
    print("YEARLY STATISTICS")
    print("="*60)
    print(yearly_stats.round(2))
    
    # COVID-19 impact note
    if 2020 in yearly_stats.index:
        print("\n⚠ Note: 2020 may show anomalous patterns due to COVID-19 pandemic")

In [ ]:
# =============================================================================
# 5.5 Autocorrelation Analysis
# =============================================================================

if target_col in df_ts.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    target_series = df_ts[target_col].dropna()
    
    # ACF
    plot_acf(target_series, lags=60, ax=axes[0], alpha=0.05)
    axes[0].set_title('Autocorrelation Function (ACF)')
    axes[0].set_xlabel('Lag (days)')
    axes[0].axvline(x=7, color='red', linestyle='--', alpha=0.5, label='Weekly (lag 7)')
    axes[0].axvline(x=30, color='green', linestyle='--', alpha=0.5, label='Monthly (lag 30)')
    axes[0].legend()
    
    # PACF
    plot_pacf(target_series, lags=30, ax=axes[1], alpha=0.05, method='ywm')
    axes[1].set_title('Partial Autocorrelation Function (PACF)')
    axes[1].set_xlabel('Lag (days)')
    
    plt.tight_layout()
    plt.savefig('figures/autocorrelation.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Numerical ACF values
    acf_values = acf(target_series, nlags=14, fft=True)
    print("\n" + "="*60)
    print("AUTOCORRELATION COEFFICIENTS")
    print("="*60)
    print("\nLag | ACF     | Interpretation")
    print("-" * 40)
    interpretations = {
        1: "Yesterday's arrivals",
        7: "Same day last week",
        14: "Two weeks ago"
    }
    for lag in [1, 2, 3, 7, 14]:
        interp = interpretations.get(lag, "")
        print(f"{lag:3d} | {acf_values[lag]:.4f}  | {interp}")
    
    print("\n" + "="*60)
    print("THESIS INSIGHT: Autocorrelation")
    print("="*60)
    print("""
    ACF Interpretation for Feature Engineering:
    - Significant lag-1: Include ED_1 (yesterday's count) as feature
    - Significant lag-7: Include ED_7 (same day last week) as feature
    - Gradual ACF decay: Suggests ARIMA(p,d,0) or moving average approach
    - Sharp PACF cutoff at lag-p: AR(p) component appropriate
    
    Recommended lag features: ED_1, ED_2, ED_3, ED_7
    """)

In [ ]:
# =============================================================================
# 5.6 Ljung-Box Test for Serial Correlation
# =============================================================================

if target_col in df_ts.columns:
    target_series = df_ts[target_col].dropna()
    
    # Ljung-Box test at multiple lags
    lags_to_test = [7, 14, 21, 30]
    
    print("\n" + "="*60)
    print("LJUNG-BOX TEST FOR SERIAL CORRELATION")
    print("="*60)
    print("\nH0: No serial correlation up to lag k")
    print("H1: Serial correlation exists")
    print("\nLag | Q-Stat  | p-value  | Conclusion")
    print("-" * 50)
    
    lb_results = acorr_ljungbox(target_series, lags=lags_to_test, return_df=True)
    
    for lag in lags_to_test:
        q_stat = lb_results.loc[lag, 'lb_stat']
        p_val = lb_results.loc[lag, 'lb_pvalue']
        conclusion = "Reject H0 (correlated)" if p_val < 0.05 else "Fail to reject H0"
        print(f"{lag:3d} | {q_stat:8.2f} | {p_val:.4e} | {conclusion}")

In [ ]:
# =============================================================================
# 5.7 Seasonal Decomposition
# =============================================================================

if target_col in df_ts.columns:
    target_series = df_ts[target_col].dropna()
    
    # STL decomposition (robust to outliers)
    stl = STL(target_series, period=7, robust=True)  # Weekly seasonality
    result = stl.fit()
    
    fig, axes = plt.subplots(4, 1, figsize=(14, 12))
    
    # Original
    axes[0].plot(target_series.index, target_series.values, linewidth=0.5)
    axes[0].set_title('Original Series')
    axes[0].set_ylabel('Arrivals')
    
    # Trend
    axes[1].plot(result.trend.index, result.trend.values, linewidth=1, color='orange')
    axes[1].set_title('Trend Component')
    axes[1].set_ylabel('Trend')
    
    # Seasonal
    axes[2].plot(result.seasonal.index, result.seasonal.values, linewidth=0.5, color='green')
    axes[2].set_title('Seasonal Component (Weekly, period=7)')
    axes[2].set_ylabel('Seasonal')
    
    # Residual
    axes[3].scatter(result.resid.index, result.resid.values, s=1, alpha=0.5, color='red')
    axes[3].axhline(y=0, color='black', linestyle='--')
    axes[3].set_title('Residual Component')
    axes[3].set_ylabel('Residual')
    axes[3].set_xlabel('Date')
    
    plt.tight_layout()
    plt.savefig('figures/seasonal_decomposition.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Decomposition statistics
    print("\n" + "="*60)
    print("SEASONAL DECOMPOSITION STATISTICS")
    print("="*60)
    print(f"\nVariance Contribution:")
    total_var = target_series.var()
    trend_var = result.trend.var()
    seasonal_var = result.seasonal.var()
    resid_var = result.resid.var()
    
    print(f"  Total variance: {total_var:.2f}")
    print(f"  Trend variance: {trend_var:.2f} ({100*trend_var/total_var:.1f}%)")
    print(f"  Seasonal variance: {seasonal_var:.2f} ({100*seasonal_var/total_var:.1f}%)")
    print(f"  Residual variance: {resid_var:.2f} ({100*resid_var/total_var:.1f}%)")
    
    # Strength of seasonality
    F_seasonal = max(0, 1 - resid_var / (seasonal_var + resid_var))
    print(f"\nStrength of seasonality: {F_seasonal:.3f}")
    print(f"Interpretation: {'Strong' if F_seasonal > 0.6 else 'Moderate' if F_seasonal > 0.3 else 'Weak'} weekly pattern")

---

## 6. Stationarity Analysis

### Justification for ML Modeling

Stationarity is fundamental for time-series modeling:
1. **ARIMA/SARIMAX** - Require stationarity for valid inference
2. **ML models** - Non-stationary data can cause concept drift
3. **Feature engineering** - Differencing may be needed to stabilize variance
4. **Forecasting validity** - Non-stationary patterns limit extrapolation

In [ ]:
# =============================================================================
# 6.1 Augmented Dickey-Fuller Test
# =============================================================================

def adf_test(series, name="Series"):
    """Perform ADF test and interpret results."""
    result = adfuller(series.dropna(), autolag='AIC')
    
    print(f"\n{'='*60}")
    print(f"Augmented Dickey-Fuller Test: {name}")
    print(f"{'='*60}")
    print(f"Test Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4e}")
    print(f"Lags Used: {result[2]}")
    print(f"Observations: {result[3]}")
    print(f"\nCritical Values:")
    for key, value in result[4].items():
        print(f"  {key}: {value:.4f}")
    
    # Interpretation
    if result[1] < 0.05:
        print(f"\n✓ CONCLUSION: Reject H0 - Series is STATIONARY (p < 0.05)")
        return True
    else:
        print(f"\n⚠ CONCLUSION: Fail to reject H0 - Series is NON-STATIONARY (p >= 0.05)")
        return False

if target_col in df_ts.columns:
    is_stationary = adf_test(df_ts[target_col], "Patient Arrivals")

In [ ]:
# =============================================================================
# 6.2 KPSS Test (Complementary)
# =============================================================================

def kpss_test(series, name="Series"):
    """Perform KPSS test and interpret results."""
    result = kpss(series.dropna(), regression='c', nlags='auto')
    
    print(f"\n{'='*60}")
    print(f"KPSS Test: {name}")
    print(f"{'='*60}")
    print(f"Test Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print(f"Lags Used: {result[2]}")
    print(f"\nCritical Values:")
    for key, value in result[3].items():
        print(f"  {key}: {value:.4f}")
    
    # Interpretation (KPSS: H0 is stationarity)
    if result[1] < 0.05:
        print(f"\n⚠ CONCLUSION: Reject H0 - Series is NON-STATIONARY (p < 0.05)")
        return False
    else:
        print(f"\n✓ CONCLUSION: Fail to reject H0 - Series is STATIONARY (p >= 0.05)")
        return True

if target_col in df_ts.columns:
    is_stationary_kpss = kpss_test(df_ts[target_col], "Patient Arrivals")

In [ ]:
# =============================================================================
# 6.3 Stationarity Test Interpretation
# =============================================================================

print("\n" + "="*60)
print("COMBINED STATIONARITY INTERPRETATION")
print("="*60)
print("""
Test Combination Guide:

ADF: H0 = Unit root (non-stationary)
KPSS: H0 = Stationary

| ADF Result  | KPSS Result | Conclusion                      |
|-------------|-------------|----------------------------------|
| Reject H0   | Don't Reject| Stationary                      |
| Don't Reject| Reject H0   | Non-stationary                  |
| Reject H0   | Reject H0   | Trend-stationary (detrend)      |
| Don't Reject| Don't Reject| Inconclusive (more tests needed)|

If non-stationary:
- Apply differencing (d=1 in ARIMA)
- Use first differences as features for ML models
- Consider detrending for trend-stationary series
""")

---

## 7. Correlation Analysis

### Justification for ML Modeling

Understanding feature correlations is critical for:
1. **Feature selection** - Identify predictive features for the target
2. **Multicollinearity detection** - Avoid redundant correlated features
3. **Hypothesis generation** - Discover unexpected relationships
4. **Model interpretation** - Understand which factors drive predictions

In [ ]:
# =============================================================================
# 7.1 Merge Datasets for Correlation Analysis
# =============================================================================

# Prepare each dataset with date as key
def prepare_for_merge(df, name):
    df_temp = df.copy()
    df_temp['date'] = pd.to_datetime(df_temp['date'], errors='coerce')
    df_temp = df_temp.dropna(subset=['date'])
    df_temp['date'] = df_temp['date'].dt.date
    return df_temp

df_patient_m = prepare_for_merge(df_patient, 'patient')
df_weather_m = prepare_for_merge(df_weather, 'weather')
df_calendar_m = prepare_for_merge(df_calendar, 'calendar')
df_clinical_m = prepare_for_merge(df_clinical, 'clinical')

# Merge all datasets
df_merged = df_patient_m.merge(df_weather_m, on='date', how='left', suffixes=('', '_weather'))
df_merged = df_merged.merge(df_calendar_m, on='date', how='left', suffixes=('', '_cal'))
df_merged = df_merged.merge(df_clinical_m, on='date', how='left', suffixes=('', '_clin'))

print(f"Merged dataset: {df_merged.shape[0]:,} rows × {df_merged.shape[1]} columns")

In [ ]:
# =============================================================================
# 7.2 Weather-Arrivals Correlation
# =============================================================================

# Select key weather features
weather_features = ['temp_mean_C', 'temp_max_C', 'temp_min_C', 'precipitation_mm', 
                    'wind_max_kmh', 'humidity_pct']
weather_features = [c for c in weather_features if c in df_merged.columns]

if target_col in df_merged.columns and weather_features:
    print("\n" + "="*60)
    print("WEATHER-ARRIVALS CORRELATION")
    print("="*60)
    print("\nPearson Correlation (linear relationship):")
    print("-" * 50)
    
    correlations = []
    for feat in weather_features:
        data = df_merged[[target_col, feat]].dropna()
        if len(data) > 30:
            r, p = pearsonr(data[target_col], data[feat])
            correlations.append({'Feature': feat, 'Pearson r': r, 'p-value': p})
            sig = "*" if p < 0.05 else ""
            print(f"{feat:25s}: r = {r:+.4f}, p = {p:.4e} {sig}")
    
    print("\n* Significant at p < 0.05")
    
    # Spearman correlation (monotonic relationship)
    print("\nSpearman Correlation (monotonic relationship):")
    print("-" * 50)
    
    for feat in weather_features:
        data = df_merged[[target_col, feat]].dropna()
        if len(data) > 30:
            rho, p = spearmanr(data[target_col], data[feat])
            sig = "*" if p < 0.05 else ""
            print(f"{feat:25s}: ρ = {rho:+.4f}, p = {p:.4e} {sig}")

In [ ]:
# =============================================================================
# 7.3 Correlation Scatter Plots (Weather vs Arrivals)
# =============================================================================

if target_col in df_merged.columns and weather_features:
    n_features = len(weather_features)
    n_cols = min(3, n_features)
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    
    for i, feat in enumerate(weather_features):
        data = df_merged[[target_col, feat]].dropna()
        
        axes[i].scatter(data[feat], data[target_col], alpha=0.3, s=10)
        
        # Regression line
        z = np.polyfit(data[feat], data[target_col], 1)
        p = np.poly1d(z)
        x_line = np.linspace(data[feat].min(), data[feat].max(), 100)
        axes[i].plot(x_line, p(x_line), 'r--', linewidth=2)
        
        # Correlation annotation
        r, _ = pearsonr(data[target_col], data[feat])
        axes[i].annotate(f'r = {r:.3f}', xy=(0.05, 0.95), xycoords='axes fraction',
                        fontsize=12, ha='left', va='top',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        axes[i].set_xlabel(feat)
        axes[i].set_ylabel(target_col)
        axes[i].set_title(f'{feat} vs Arrivals')
    
    # Hide unused subplots
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.savefig('figures/weather_correlation_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# =============================================================================
# 7.4 Holiday Effect Analysis
# =============================================================================

# Find holiday column
holiday_cols = [c for c in df_merged.columns if 'holiday' in c.lower() or 'public' in c.lower()]

if target_col in df_merged.columns and holiday_cols:
    print("\n" + "="*60)
    print("HOLIDAY EFFECT ANALYSIS")
    print("="*60)
    
    for hol_col in holiday_cols[:3]:  # Analyze first 3
        data = df_merged[[target_col, hol_col]].dropna()
        
        # Group by holiday status
        groups = data.groupby(hol_col)[target_col].agg(['mean', 'std', 'count'])
        
        print(f"\n{hol_col}:")
        print(groups.round(2))
        
        # T-test for difference
        if len(groups) == 2:
            group0 = data[data[hol_col] == 0][target_col]
            group1 = data[data[hol_col] == 1][target_col]
            t_stat, p_val = stats.ttest_ind(group0, group1)
            print(f"T-test: t = {t_stat:.3f}, p = {p_val:.4e}")
            
            effect_size = (group1.mean() - group0.mean()) / group0.mean() * 100
            print(f"Effect size: {effect_size:+.1f}% {'(significant)' if p_val < 0.05 else ''}")

In [ ]:
# =============================================================================
# 7.5 Feature Correlation Heatmap
# =============================================================================

# Select key features for heatmap
key_features = [target_col] + weather_features[:4]  # Target + top weather

# Add calendar features if available
cal_features = ['day_of_week', 'is_weekend', 'is_public_holiday', 'month']
cal_features = [c for c in cal_features if c in df_merged.columns]
key_features.extend(cal_features[:3])

# Filter to existing columns
key_features = [c for c in key_features if c in df_merged.columns]

if len(key_features) > 2:
    corr_matrix = df_merged[key_features].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax, square=True,
                cbar_kws={'label': 'Correlation Coefficient'})
    
    ax.set_title('Feature Correlation Matrix', fontsize=14)
    
    plt.tight_layout()
    plt.savefig('figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*60)
    print("THESIS INSIGHT: Correlation Analysis")
    print("="*60)
    print("""
    Key Considerations:
    1. Features highly correlated with target are good predictors
    2. Features highly correlated with each other introduce multicollinearity
    3. Consider removing one of highly correlated feature pairs (r > 0.8)
    4. Temperature variables often correlated - choose one representative
    5. Create interaction terms for weakly correlated but conceptually important features
    """)

---

## 8. Outlier Detection

### Justification for ML Modeling

Outliers can significantly impact model performance:
1. **Training bias** - Outliers can skew parameter estimates
2. **Evaluation distortion** - MAE/RMSE are sensitive to extreme values
3. **Real events vs errors** - True extreme events (holidays) should be retained
4. **Robust modeling** - Some algorithms are more outlier-resistant

In [ ]:
# =============================================================================
# 8.1 IQR-Based Outlier Detection
# =============================================================================

def detect_outliers_iqr(series, multiplier=1.5):
    """Detect outliers using IQR method."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    
    outliers = (series < lower_bound) | (series > upper_bound)
    
    return outliers, lower_bound, upper_bound

if target_col in df_ts.columns:
    target_series = df_ts[target_col].dropna()
    
    outliers, lb, ub = detect_outliers_iqr(target_series)
    
    print("\n" + "="*60)
    print("IQR OUTLIER DETECTION")
    print("="*60)
    print(f"Lower bound: {lb:.2f}")
    print(f"Upper bound: {ub:.2f}")
    print(f"Number of outliers: {outliers.sum()} ({100*outliers.mean():.2f}%)")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Time series with outliers highlighted
    axes[0].plot(target_series.index, target_series.values, linewidth=0.5, alpha=0.7)
    axes[0].scatter(target_series.index[outliers], target_series[outliers], 
                    color='red', s=20, label=f'Outliers (n={outliers.sum()})')
    axes[0].axhline(y=ub, color='red', linestyle='--', alpha=0.5, label=f'Upper bound ({ub:.0f})')
    axes[0].axhline(y=lb, color='red', linestyle='--', alpha=0.5, label=f'Lower bound ({lb:.0f})')
    axes[0].set_title('Time Series with IQR Outliers')
    axes[0].legend()
    
    # Box plot
    bp = axes[1].boxplot(target_series, vert=True, patch_artist=True)
    bp['boxes'][0].set_facecolor('steelblue')
    axes[1].set_title('Box Plot: Outlier Visualization')
    axes[1].set_ylabel('Patient Count')
    
    plt.tight_layout()
    plt.savefig('figures/outlier_detection.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# =============================================================================
# 8.2 Contextual Outlier Analysis
# =============================================================================

if target_col in df_ts.columns:
    outlier_dates = df_ts[outliers].copy()
    
    if len(outlier_dates) > 0:
        print("\n" + "="*60)
        print("CONTEXTUAL ANALYSIS OF OUTLIERS")
        print("="*60)
        
        # Day of week distribution of outliers
        dow_outliers = outlier_dates['day_name'].value_counts()
        print("\nOutliers by Day of Week:")
        print(dow_outliers)
        
        # Month distribution of outliers
        if 'month_name' in outlier_dates.columns:
            month_outliers = outlier_dates['month_name'].value_counts()
            print("\nOutliers by Month:")
            print(month_outliers.head(5))
        
        # Top 10 highest arrival days
        print("\nTop 10 Highest Arrival Days:")
        top_10 = df_ts.nlargest(10, target_col)[[target_col, 'day_name']]
        for idx, row in top_10.iterrows():
            print(f"  {idx.strftime('%Y-%m-%d')} ({row['day_name'][:3]}): {row[target_col]:.0f} arrivals")
    
    print("\n" + "="*60)
    print("THESIS INSIGHT: Outlier Handling")
    print("="*60)
    print("""
    Recommended Strategies:
    1. Investigate top outliers - are they holidays/events or data errors?
    2. For real events: Keep data, add binary indicator feature
    3. For data errors: Impute using rolling median or remove
    4. Consider robust models (e.g., XGBoost with Huber loss)
    5. Winsorize extreme values (cap at 1st/99th percentile) for sensitive models
    6. Document all outlier handling decisions for reproducibility
    """)

---

## 9. Feature Engineering Recommendations

Based on the EDA findings, this section provides recommendations for feature engineering to support forecasting model development.

In [ ]:
# =============================================================================
# 9.1 Summary of EDA Findings
# =============================================================================

print("\n" + "="*60)
print("FEATURE ENGINEERING RECOMMENDATIONS")
print("="*60)

print("""
Based on the exploratory data analysis, the following features are recommended
for the forecasting model:

1. LAG FEATURES (from autocorrelation analysis):
   - ED_1: Yesterday's patient count
   - ED_2: Two days ago
   - ED_3: Three days ago
   - ED_7: Same day last week (weekly pattern)
   - Rolling mean (7-day, 14-day)
   - Rolling std (7-day) for volatility

2. CALENDAR FEATURES (from temporal analysis):
   - day_of_week: Strong weekly pattern identified
   - month: Monthly seasonality present
   - is_weekend: Binary weekend indicator
   - is_holiday: Public holiday indicator
   - is_day_before_holiday: Pre-holiday effect
   - is_payday: End-of-month effect (25th-31st)
   - is_school_holiday: School vacation periods

3. WEATHER FEATURES (from correlation analysis):
   - temperature: Mean/max temperature
   - precipitation: Rainfall indicator
   - Consider lagged weather (t-1, t-2) for delayed effects

4. CYCLICAL ENCODING (for periodic features):
   - day_of_week_sin, day_of_week_cos
   - month_sin, month_cos
   - day_of_year_sin, day_of_year_cos

5. INTERACTION FEATURES:
   - weekend × is_holiday
   - temperature × month (seasonal temperature effect)
   - rain × winter (respiratory illness trigger)

6. TARGET TRANSFORMATIONS:
   - Consider log transformation if target is right-skewed
   - Use first differences if non-stationary trend detected
""")

In [ ]:
# =============================================================================
# 9.2 Model Selection Guidance
# =============================================================================

print("\n" + "="*60)
print("MODEL SELECTION GUIDANCE")
print("="*60)

print("""
Based on the EDA findings, consider the following modeling approaches:

1. BASELINE MODELS:
   - Naive (yesterday's value)
   - Seasonal Naive (same day last week)
   - Moving Average (7-day)

2. STATISTICAL MODELS:
   - ARIMA: If ACF shows gradual decay
   - SARIMA: For weekly seasonality (period=7)
   - SARIMAX: Include exogenous weather/calendar features

3. MACHINE LEARNING MODELS:
   - XGBoost: Robust to outliers, handles non-linear relationships
   - Random Forest: Good baseline, feature importance
   - LightGBM: Fast training, handles categorical features

4. DEEP LEARNING MODELS:
   - LSTM: Captures long-term dependencies
   - GRU: Lighter alternative to LSTM
   - Transformer: For very long sequences

5. HYBRID APPROACHES:
   - LSTM + XGBoost residual correction
   - SARIMA for trend + ML for residuals

6. EVALUATION METRICS:
   - MAE: Interpretable, robust to outliers
   - RMSE: Penalizes large errors
   - MAPE: Percentage error (if no zeros)
   - SMAPE: Symmetric percentage error
   - Use time-series cross-validation (rolling window)
""")

---

## 10. Conclusion and Next Steps

### Summary of Key Findings

This EDA has revealed several important patterns in the Steve Biko Hospital ED arrival data:

1. **Temporal Patterns**: Strong weekly seasonality with different arrival patterns on weekdays vs weekends
2. **Seasonal Effects**: Monthly variation with potential winter peaks (respiratory conditions)
3. **Autocorrelation**: Significant lag-1 and lag-7 correlations support inclusion of lag features
4. **Weather Influence**: Moderate correlations with temperature variables
5. **Holiday Effects**: Distinct patterns during public holidays
6. **Trauma Burden**: High proportion of trauma and violence cases (South African context)

### Implications for Staff and Inventory Planning

- **Staff Scheduling**: Higher staffing on weekends and during winter months
- **Trauma Resources**: Ensure adequate trauma supplies given high trauma burden
- **Seasonal Preparation**: Stock respiratory medications before winter
- **Holiday Planning**: Adjust resources for public holiday patterns

### Next Steps

1. Feature engineering based on EDA insights
2. Train-validation-test split (70-15-15)
3. Baseline model establishment
4. Advanced model training and tuning
5. Model comparison and selection
6. Integration with HealthForecast AI platform

In [ ]:
# =============================================================================
# 10.1 Save Processed Data for Modeling
# =============================================================================

# Create figures directory if needed
import os
os.makedirs('figures', exist_ok=True)

# Save merged dataset for modeling
output_path = '../data/steve_biko_merged_eda.csv'
df_merged.to_csv(output_path, index=False)
print(f"\nMerged dataset saved to: {output_path}")
print(f"Shape: {df_merged.shape[0]:,} rows × {df_merged.shape[1]} columns")

print("\n" + "="*60)
print("EDA COMPLETE")
print("="*60)
print("\nAll figures saved to: notebooks/figures/")
print("Merged data saved for model training")
print("\nProceed to Feature Engineering and Model Training stages.")